# 07 - Feature Engineering

Five models that each rank the full catalog on their own logic. This phase
starts a different approach: instead of hand-weighting model outputs like the hybrid did,
train a model to learn the ranking directly from labeled (user, product) pairs.

That needs two things before any training can happen:

1. **Candidate generation** — scoring all ~50k products per user would be wasteful and mostly
   pointless, so each user gets a shortlist: their own prior purchases, plus Personalized
   Frequency's and ALS's top picks. This reuses Phase 5's models as a first-pass filter
   rather than throwing that work away.
2. **Features** — for every (user, candidate product) pair, a set of numeric features
   describing the user, the product, and their specific interaction history. Everything here
   is computed from `eval_set == 'prior'` only, never `train` — `train` is where the label
   comes from, and letting it leak into the features would make any offline evaluation of
   the resulting model meaningless.

The label is binary: did this candidate actually show up in the user's real train basket.

Logic in `src/candidate_generation.py` and `src/feature_engineering.py`.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd

from data_processing import load_raw_data
from eda import build_transactions
from baseline_models import PersonalizedFrequencyModel, get_eval_users
from collaborative_filtering import ALSModel
from candidate_generation import generate_candidates_bulk
from feature_engineering import build_training_table, add_features

## Load data

In [2]:
processed_path = Path.cwd().parent / "data" / "processed" / "transactions.parquet"

if processed_path.exists():
    txn = pd.read_parquet(processed_path)
else:
    data = load_raw_data()
    txn = build_transactions(data)

eval_users = get_eval_users(txn)
print(txn.shape, len(eval_users), "eval users")

(33819106, 15) 131209 eval users


## Fit the candidate-generating models

Personalized Frequency and ALS from Phase 5, same settings as the hybrid — `alpha=15`.

In [3]:
pf_model = PersonalizedFrequencyModel(top_n=50).fit(txn)
als_model = ALSModel(factors=50, regularization=0.01, alpha=15.0, iterations=15).fit(txn)

C:\Users\shubh\AppData\Roaming\Python\Python314\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

## Generate candidates

Each user's shortlist: their own prior purchases + top-50 from each model. Union, not
concatenation, so a product surfaced by more than one source only appears once.

In [4]:
candidates = generate_candidates_bulk(eval_users, txn, pf_model, als_model, n_each=50)

candidate_sizes = pd.Series({uid: len(c) for uid, c in candidates.items()})
print(candidate_sizes.describe())

count    131209.000000
mean        102.976153
std          53.255475
min          50.000000
25%          67.000000
50%          86.000000
75%         121.000000
max         754.000000
dtype: float64


## Build the training table

One row per (user, candidate product), with a binary label — did the product actually appear
in that user's train basket.

In [5]:
table = build_training_table(candidates, txn, eval_users)
print(table.shape)
print("label distribution:", table["label"].value_counts(normalize=True).to_dict())

(13511398, 3)
label distribution: {0: 0.9338132886027042, 1: 0.06618671139729583}


The positive rate here is the class balance the ranking model will train on — worth noting,
since a heavily skewed label distribution usually means the model needs
class weighting or a ranking-aware loss rather than plain binary cross-entropy.

## Add features

User-level (order frequency, basket size, reorder rate), product-level (purchase volume,
reorder rate, typical cart position), and user-product interaction (times bought, orders
since last bought). Candidates with no matching history — a new user, or a product the
user's never touched — are filled to 0 rather than dropped, since dropping them would
silently shrink coverage for exactly the cases a real system still needs to serve something to.

In [6]:
featured = add_features(table, txn)
print(featured.shape)
print("nulls:", featured.isnull().sum().sum())
featured.head()

(13511398, 15)
nulls: 0


,user_id,product_id,label,n_orders,avg_basket_size,avg_days_between_orders,reorder_rate,n_purchases,product_reorder_rate,avg_add_to_cart_position,up_purchase_count,up_last_order_number,up_avg_cart_position,user_max_order_number,orders_since_last_purchase
0,1,11266,0,10,5.9,20.25926,0.694915,4081,0.735114,4.667238,0.0,0.0,0.0,10,0.0
1,1,14084,0,10,5.9,20.25926,0.694915,15935,0.810982,5.792595,1.0,1.0,2.0,10,9.0
2,1,42500,0,10,5.9,20.25926,0.694915,4053,0.618801,4.212929,0.0,0.0,0.0,10,0.0
3,1,13575,0,10,5.9,20.25926,0.694915,12575,0.778926,3.757932,0.0,0.0,0.0,10,0.0
4,1,40199,0,10,5.9,20.25926,0.694915,14148,0.602276,6.884577,0.0,0.0,0.0,10,0.0


### Findings

The feature-engineering phase converts the outputs of the earlier recommendation models into a structured learning-to-rank dataset. Rather than hand-weighting model outputs as in the hybrid approach, this phase prepares user-product candidate pairs and behavioral features that can be used by a ranking model to learn the final ordering directly.

* **Candidate generation is computationally efficient and provides broad coverage.** Each evaluation user's candidates are formed from their prior purchases together with the top-50 recommendations from Personalized Frequency and ALS. The three sources are combined using a union, preventing duplicate products from appearing multiple times. The resulting candidate set contains an average of **103 products per user**, with a median of **86**, while the maximum is **754**. This is substantially smaller than ranking the full product catalog for every user and provides a practical candidate space for the ranking model.

* **The training data is strongly class-imbalanced.** The resulting training table contains **13,511,398 user-product candidate pairs**, of which approximately **6.62% are positive examples** and **93.38% are negative examples**. This imbalance should be explicitly considered when selecting the ranking objective and training strategy; optimizing a standard classification loss without accounting for the imbalance could favor negative predictions.

* **The feature set combines multiple levels of behavioral information.** The engineered variables capture **user-level behavior** such as order frequency, basket size, order interval, and reorder rate; **product-level behavior** such as purchase volume and product reorder rate; and **user-product interaction history** such as purchase frequency, previous cart position, and recency since the last purchase. This provides substantially richer information than the standalone popularity, frequency, ALS, or content-based models.

* **Historical information is kept separate from the evaluation label.** All features are generated from `eval_set == 'prior'`, while the user's train basket is used only to construct the binary target. This preserves the offline evaluation setup established in the earlier phases and prevents target leakage into the feature set.

* **Missing interaction history is handled explicitly.** Candidates with no prior user-product interaction are retained and their corresponding history-based features are filled with zero rather than being removed. This preserves candidate coverage and allows the ranking model to distinguish previously purchased products from candidates with no direct interaction history.

* **The feature table is complete and ready for model training.** The final dataset contains **13,511,398 rows and 15 columns**, with **no missing values** across the engineered features. This provides a consistent training matrix for the ranking model in the next phase.

* **Modeling implication:** The feature-engineering results support moving from manually designed recommendation combinations toward a **learning-to-rank model**. The strong class imbalance suggests that the next phase should use an appropriate ranking objective or class-weighting strategy, while the candidate-set distribution indicates that the current candidate-generation approach provides a manageable and sufficiently broad search space.


In [7]:
featured.to_parquet(
    Path.cwd().parent / "data" / "processed" / "ranking_training_table.parquet", index=False
)